In [ ]:
import csv
from datetime import datetime
from decimal import Decimal
from pathlib import Path

In [ ]:
# Bakery

DATASET = "BAKERY"

BASE_DIR = Path(".")
OUTPUT_DIR = BASE_DIR

FILES = {
    "CUSTOMERS": BASE_DIR / "customers.csv",
    "GOODS":     BASE_DIR / "goods.csv",
    "RECEIPTS":  BASE_DIR / "receipts.csv",
    "ITEMS":     BASE_DIR / "items.csv",
}

def strip_outer_single_quotes(s):
    if s is None:
        return None
    s = s.strip()
    if len(s) >= 2 and s[0] == "'" and s[-1] == "'":
        return s[1:-1]
    return s

def sql_string(s):
    if s is None:
        return "NULL"
    s = strip_outer_single_quotes(s)
    if s == "":
        return "NULL"
    q = "'"
    return f"'{s.replace(q, q + q)}'"

def sql_int(s):
    s = s.strip()
    if s == "":
        return "NULL"
    return str(int(s))

def sql_decimal(s):
    s = s.strip()
    if s == "":
        return "NULL"
    return f"{Decimal(s):.2f}"

def sql_date(s):
    s = strip_outer_single_quotes(s).strip()
    dt = datetime.strptime(s, "%d-%b-%Y")   # big date time yea man
    return f"'{dt.strftime('%Y-%m-%d')}'"

# customer
with open(FILES["CUSTOMERS"], newline="", encoding="utf-8") as fin, \
     open(OUTPUT_DIR / f"{DATASET}-build-customers.sql", "w", encoding="utf-8") as fout:

    r = csv.reader(fin, skipinitialspace=True)
    next(r)
    

    for row in r:
        fout.write(
            f"INSERT INTO CUSTOMERS VALUES ("
            f"{sql_int(row[0])}, "
            f"{sql_string(row[1])}, "
            f"{sql_string(row[2])}"
            f");\n"
        )

# goods
with open(FILES["GOODS"], newline="", encoding="utf-8") as fin, \
     open(OUTPUT_DIR / f"{DATASET}-build-goods.sql", "w", encoding="utf-8") as fout:

    r = csv.reader(fin, skipinitialspace=True)
    next(r)
    

    for row in r:
        fout.write(
            f"INSERT INTO GOODS VALUES ("
            f"{sql_string(row[0])}, "
            f"{sql_string(row[1])}, "
            f"{sql_string(row[2])}, "
            f"{sql_decimal(row[3])}"
            f");\n"
        )

# receipts
with open(FILES["RECEIPTS"], newline="", encoding="utf-8") as fin, \
     open(OUTPUT_DIR / f"{DATASET}-build-receipts.sql", "w", encoding="utf-8") as fout:

    r = csv.reader(fin, skipinitialspace=True)
    next(r)
    

    for row in r:
        fout.write(
            f"INSERT INTO RECEIPTS VALUES ("
            f"{sql_int(row[0])}, "
            f"{sql_date(row[1])}, "
            f"{sql_int(row[2])}"
            f");\n"
        )

# items
with open(FILES["ITEMS"], newline="", encoding="utf-8") as fin, \
     open(OUTPUT_DIR / f"{DATASET}-build-items.sql", "w", encoding="utf-8") as fout:

    r = csv.reader(fin, skipinitialspace=True)
    next(r)
    

    for row in r:
        fout.write(
            f"INSERT INTO ITEMS VALUES ("
            f"{sql_int(row[0])}, "
            f"{sql_int(row[1])}, "
            f"{sql_string(row[2])}"
            f");\n"
        )

In [15]:
# Marathon

DATASET = "MARATHON"

BASE_DIR = Path(".")
INPUT_CSV = BASE_DIR / "marathon.csv"
OUTPUT_SQL = BASE_DIR / f"{DATASET}-build-marathon.sql"

def strip_outer_single_quotes(s):
    if s is None:
        return None
    s = s.strip()
    if len(s) >= 2 and s[0] == "'" and s[-1] == "'":
        return s[1:-1]
    return s

def sql_string(s):
    if s is None:
        return "NULL"
    s = strip_outer_single_quotes(s)
    s = s.strip()
    if s == "":
        return "NULL"
    q = "'"
    return f"'{s.replace(q, q + q)}'"

def sql_int(s):
    if s is None:
        return "NULL"
    s = s.strip()
    if s == "":
        return "NULL"
    return str(int(float(s)))

def sql_char(s):
    if s is None:
        return "NULL"
    s = strip_outer_single_quotes(s).strip()
    if s == "":
        return "NULL"
    return f"'{s[0]}'"

def sql_time(s):
    """Normalize TIME values to 'HH:MM:SS' and return quoted SQL literal, or NULL."""
    if s is None:
        return "NULL"
    s = strip_outer_single_quotes(s).strip()
    if s == "":
        return "NULL"
    # Try several common formats
    fmts = ["%H:%M:%S", "%H:%M", "%I:%M:%S", "%I:%M"]
    for fmt in fmts:
        try:
            dt = datetime.strptime(s, fmt)
            return f"'{dt.strftime('%H:%M:%S')}'"
        except Exception:
            pass
    # If given mm:ss (pace accidentally) convert to 00:mm:ss
    parts = s.split(':')
    if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
        return f"'00:{parts[0].zfill(2)}:{parts[1].zfill(2)}'"
    raise ValueError(f"Unrecognized time format: {s!r}")


with open(INPUT_CSV, newline="", encoding="utf-8") as fin, \
     open(OUTPUT_SQL, "w", encoding="utf-8") as fout:

    reader = csv.reader(fin, skipinitialspace=True)
    _ = next(reader, None)  # skip header row if present

    for row in reader:
        # Expect columns: Place, Time, Pace, GroupPlace, Group, Age, Sex, BIBNumber, FirstName, LastName, Town, State
        # Defensive padding if row has fewer columns
        while len(row) < 12:
            row.append("")

        place      = sql_int(row[0])
        time       = sql_time(row[1])
        pace       = sql_string(row[2])
        groupplace = sql_int(row[3])
        group_name = sql_string(row[4])
        age        = sql_int(row[5])
        sex        = sql_char(row[6])
        bib        = sql_int(row[7])
        firstname  = sql_string(row[8])
        lastname   = sql_string(row[9])
        town       = sql_string(row[10])
        state      = sql_string(row[11])

        fout.write(
            "INSERT INTO MARATHON VALUES ("
            f"{place}, {time}, {pace}, {groupplace}, {group_name}, {age}, {sex}, {bib}, {firstname}, {lastname}, {town}, {state}"
            ");\n"
        )

In [16]:
# INN

DATASET = "INN"

BASE_DIR = Path(".")
ROOMS_CSV = BASE_DIR / "Rooms.csv"
RES_CSV   = BASE_DIR / "Reservations.csv"

ROOMS_SQL = BASE_DIR / f"{DATASET}-build-rooms.sql"
RES_SQL   = BASE_DIR / f"{DATASET}-build-reservations.sql"

def strip_quotes(s):
    if s is None:
        return None
    s = s.strip()
    if len(s) >= 2 and s[0] == "'" and s[-1] == "'":
        return s[1:-1]
    return s

def sql_string(s):
    if s is None:
        return "NULL"
    s = strip_quotes(s).strip()
    if s == "":
        return "NULL"
    q = "'"
    return f"'{s.replace(q, q + q)}'"

def sql_int(s):
    s = s.strip()
    if s == "":
        return "NULL"
    return str(int(s))

def sql_decimal(s):
    s = s.strip()
    if s == "":
        return "NULL"
    return f"{float(s):.2f}"

def sql_date(s):
    s = strip_quotes(s).strip()
    dt = datetime.strptime(s, "%d-%b-%y")   # e.g. 01-JAN-10
    return f"'{dt.strftime('%Y-%m-%d')}'"

with open(ROOMS_CSV, newline="", encoding="utf-8") as fin, \
     open(ROOMS_SQL, "w", encoding="utf-8") as fout:

    r = csv.reader(fin, skipinitialspace=True)
    next(r)

    for row in r:
        fout.write(
            "INSERT INTO ROOMS VALUES ("
            f"{sql_string(row[0])}, "
            f"{sql_string(row[1])}, "
            f"{sql_int(row[2])}, "
            f"{sql_string(row[3])}, "
            f"{sql_int(row[4])}, "
            f"{sql_decimal(row[5])}, "
            f"{sql_string(row[6])}"
            ");\n"
        )

with open(RES_CSV, newline="", encoding="utf-8") as fin, \
     open(RES_SQL, "w", encoding="utf-8") as fout:

    r = csv.reader(fin, skipinitialspace=True)
    next(r)

    for row in r:
        fout.write(
            "INSERT INTO RESERVATIONS VALUES ("
            f"{sql_int(row[0])}, "
            f"{sql_string(row[1])}, "
            f"{sql_date(row[2])}, "
            f"{sql_date(row[3])}, "
            f"{sql_decimal(row[4])}, "
            f"{sql_string(row[5])}, "
            f"{sql_string(row[6])}, "
            f"{sql_int(row[7])}, "
            f"{sql_int(row[8])}"
            ");\n"
        )
